# NC-Musical: AI Music Transcription & Interactive Editor
Run GPU-accelerated automatic music transcription (AMT) using the MuScriptor engine and edit transcriptions interactively in your web browser via the Piano Roll Web GUI.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentHitmanFaris/NC-Musical/blob/Stable/NC_Musical_Colab.ipynb)

In [ ]:
# @title 1. Check GPU & Setup Repository
import os
import sys

print("Checking GPU environment...")
!nvidia-smi

print("\nInstalling system dependencies (FFmpeg and FluidSynth)...")
!apt-get update -qq
!apt-get install -y -qq ffmpeg fluidsynth

print("\nCloning NC-Musical repository...")
if not os.path.exists("NC-Musical"):
    !git clone https://github.com/AgentHitmanFaris/NC-Musical.git
%cd NC-Musical

print("\nInstalling Python dependencies...")
!pip install -q uvicorn fastapi yt-dlp soundfile torch torchvision torchaudio
!pip install -q muscriptor || true

print("\nRunning patch script...")
!python patch_muscriptor.py || true


In [ ]:
# @title 2. Download MS Basic.sf3 SoundFont (if missing)
import os
import urllib.request

sf3_path = "MS Basic.sf3"
if not os.path.exists(sf3_path):
    print("Downloading MS Basic.sf3 SoundFont (~50MB)...")
    sf3_url = "https://raw.githubusercontent.com/musescore/MuseScore/master/share/sound/MS%20Basic.sf3"
    try:
        urllib.request.urlretrieve(sf3_url, sf3_path)
        print("MS Basic.sf3 downloaded successfully!")
    except Exception as e:
        print("Download notice:", e)
else:
    print("MS Basic.sf3 SoundFont found!")


In [ ]:
# @title 3. Start Server & Open Web GUI
import os
import time
import subprocess
from google.colab import output

# Kill any existing server instance
!pkill -f "server_gui.py" || true

PORT = 8222

print("Starting MuScriptor FastAPI server on GPU...")
server_process = subprocess.Popen(
    ["python", "server_gui.py", "--port", str(PORT), "--model", "large", "--device", "cuda"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(3)

# Primary Colab Proxy Link
try:
    colab_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
    print("\n" + "="*65)
    print("SUCCESS! Click the link below to open the MuScriptor Web GUI:")
    print(f"--> {colab_url}index.html <--")
    print("="*65 + "\n")
except Exception as e:
    print("Colab proxy link notice:", e)

# Alternative Cloudflare Tunnel Link
print("Creating alternative Cloudflare Tunnel link...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

cf_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(4)
for _ in range(20):
    line = cf_process.stdout.readline()
    if "trycloudflare.com" in line:
        url = line.strip().split()[-1]
        print(f"Alternative Public Link (Cloudflare): {url}/index.html")
        break
    time.sleep(0.5)
